# Grade Distribution Around G3+ AE Episodes

For each adverse event, this shows the grade distribution of that AE's **G3+ cohort** at each
90-day window relative to the patient's *first* G3+ window (offsets pre-K..pre-1) and *last* G3+
window (offsets post-1..post-K), with the peak placed at day 0 on a true day axis.

**Cohort:** restricted to patients with LLM predictions, matching every other S2 panel.

**Follow-up scope — this panel differs deliberately.** S2A–S2D and S2F restrict to first line of
therapy; S2G follows patients across all lines. Resolution and recurrence of an irAE episode are
properties of the adverse event, not of the treatment line, and `t_cutoff_lot` truncates at
`next_lot_start` — a boundary with no bearing on whether a G3+ episode resolved. The restriction
is also infeasible here: median `t_cutoff_lot` is 243 days against the ~450 days of contiguous
follow-up a ±180-day horizon requires, leaving n = 1–11 per toxicity. Set `RESTRICT_TO_LOT1 = True`
in the constants cell to reproduce that version as a sensitivity analysis.

**Complete-follow-up restriction (important):** only patients with **observed window data at every offset** in
the display horizon (pre-1..pre-K and post-1..post-K, all present) are included. This holds the
denominator constant across offsets, so the gap from the top of the stack up to 100% is true
observed grade-0 (AE resolved / not detected) rather than loss-to-follow-up. Without this
restriction, dropout would be indistinguishable from resolution, biasing the apparent resolution
rate upward.

At each offset, the stack from bottom to top is G1, G2, G3, G4, G5 as a percent of the cohort. The
peak column is 100% by construction, since every included patient is G3+ (max grade across their
G3+ windows is ≥3).

Hyperthyroidism is dropped (too few G3+ events for a stable trajectory) — the remaining five AEs
share one 2×3 grid figure, one cell unused.


**Formatting (Nature compliance):** Arial only (hard-fails if not resolved), `pdf.fonttype=42`,
7pt panel titles/axis labels, 6pt tick labels, 5pt legend/annotations, no `bbox_inches='tight'` on
save. The original script's `fig.suptitle(...)` was decorative and has been removed — add a title
in Illustrator if needed, matching every other panel in this project.


In [ ]:
import os
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib
import re
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.sans-serif'] = ['Arial']
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
matplotlib.rcParams['axes.unicode_minus'] = False

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

%matplotlib inline

warnings.filterwarnings('ignore')

# ---- Hard-fail if Arial isn't actually resolved (no silent fallback) ----
import matplotlib.font_manager as fm
_arial_path = fm.findfont('Arial', fallback_to_default=False)
if 'Arial' not in _arial_path:
    raise RuntimeError(
        f"Arial not found -- matplotlib resolved to '{_arial_path}' instead. "
        "Install Arial or update font.sans-serif before rendering this figure."
    )
print(f"Arial resolved to: {_arial_path}")

## Paths


In [ ]:
NOTEBOOK_DIR = os.getcwd()
FIGURES_DIR = os.path.normpath(os.path.join(NOTEBOOK_DIR, '..', '..', '..'))
DATA_DIR = os.path.join(FIGURES_DIR, 'figures_data', 'figure 2', 'data')
RESULTS_DIR = os.path.normpath(os.path.join(NOTEBOOK_DIR, '..', 'results', 'supp', 'S2G_Grade_Distribution_G3+'))
os.makedirs(RESULTS_DIR, exist_ok=True)

GRADE_PATH = os.path.join(DATA_DIR, 'grade_results_84k_FIXED_FP.csv')
PDF_OUT = os.path.join(RESULTS_DIR, 'Grade_Distribution_S2G.pdf')
CSV_OUT = os.path.join(RESULTS_DIR, 'Grade_Distribution_S2G.csv')
LLM_PATIENT_PATH = os.path.join(DATA_DIR, 'llm_calls_patient_level_84k.csv')
COVARS_PATH = os.path.join(DATA_DIR, 'OneDrive_1_8-7-2026', 'llm84k_pneumonitis_grade0_20260630.csv')

for p in [GRADE_PATH, LLM_PATIENT_PATH, COVARS_PATH]:
    print(('FOUND   ' if os.path.exists(p) else 'MISSING '), p)

## Constants

`K_PRE`/`K_POST` control the horizon: `K=2` with 90-day windows → ±180 days (the published-figure
setting). `K=3` → ±270 days, smaller cohort.


In [ ]:
K_PRE = 2
K_POST = 2
WINDOW_DAYS = 90

# Follow-up scope. S2A-S2D and S2F restrict to first line of therapy; S2G deliberately does
# not -- resolution/recurrence of an irAE episode is a property of the AE, not the treatment
# line, and t_cutoff_lot truncates at next_lot_start. Median t_cutoff_lot is 243 days against
# the ~450 days of contiguous follow-up a +/-180d horizon needs, so the LOT-1 version leaves
# n = 2/3/1/5/11 per toxicity. Set True to reproduce that sensitivity analysis.
RESTRICT_TO_LOT1 = False

# Five AEs in panel order (hyperthyroidism dropped -- too few G3+ events for a stable trajectory)
TOXICITIES = ['adrenal_insufficiency', 'colitis', 'hypothyroidism', 'pneumonitis', 'liver_toxicity']
TOX_DISPLAY = {
    'adrenal_insufficiency': 'Adrenal Insufficiency', 'colitis': 'Colitis',
    'hypothyroidism': 'Hypothyroidism', 'pneumonitis': 'Pneumonitis',
    'liver_toxicity': 'Liver Toxicity',
}

# Grades stacked bottom-to-top -- matches this project's unified grade palette
GRADES_STACKED = [1, 2, 3, 4, 5]
GRADE_COLORS = {1: '#FEE391', 2: '#F4A02C', 3: '#F4663A', 4: '#B30000', 5: '#54278F'}
GRADE_LABEL = {1: 'G1', 2: 'G2', 3: 'G3', 4: 'G4', 5: 'G5'}

def standardize_mrn(mrn):
    """Project-standard MRN normalization -- first digit group, zero-padded to 8.
    NOT ''.join(isdigit) over the whole string: that concatenates every digit run, so
    '12345678 (2)' would normalize to '123456782' here and to '12345678' in every other
    notebook, silently breaking the join on exactly the rows that are formatted oddly.
    """
    if pd.isna(mrn):
        return None
    try:
        digits = re.findall(r"\d+", str(mrn).strip().strip("'\""))
        return str(int(digits[0])).zfill(8) if digits else None
    except (ValueError, TypeError):
        return None

## Load the window-level grade table

One row per (patient, 90-day window) with an integer grade (0-5) per toxicity, sorted by
`[mrn, window_start]` so each patient's grade sequence is chronological (the flow extractor below
relies on this ordering). Grade-0 windows are retained -- they form the gap to 100% in the plot.

If this file lives on OneDrive, files-on-demand sync can abort a streamed read with `OSError`
while it hydrates the placeholder -- the read is retried a few times (each attempt nudges
hydration along) before giving up with an actionable message.


In [ ]:
def load_grade_data(grade_path):
    print('Loading window-level grade table...')
    df = None
    for attempt in range(1, 6):
        try:
            df = pd.read_csv(grade_path, encoding='latin-1', low_memory=False)
            break
        except OSError as exc:
            print(f'  Read attempt {attempt} of 5 failed ({exc}); '
                  f'waiting for OneDrive to finish hydrating the file...')
            time.sleep(5)
    if df is None:
        raise OSError(
            f"Could not read {grade_path} after 5 attempts: OneDrive files-on-demand kept "
            "canceling the read. Fix: in Finder right-click the Data folder, choose "
            "'Always Keep on This Device' to force a full local download (or copy the CSV to a "
            "non-OneDrive folder), then rerun."
        )
    df['mrn'] = df['mrn'].apply(standardize_mrn)
    df = df[df['mrn'].notna()].copy()
    df['window_start'] = pd.to_datetime(df['window_start'], errors='coerce')
    df = df[df['window_start'].notna()].copy()

    # Reconcile space-separated source column names to the canonical underscore names.
    for old, new in [('adrenal insufficiency', 'adrenal_insufficiency'), ('liver toxicity', 'liver_toxicity')]:
        if old in df.columns and new not in df.columns:
            df[new] = df[old]
    for tox in TOXICITIES:
        if tox in df.columns:
            df[tox] = pd.to_numeric(df[tox], errors='coerce').fillna(0).astype(int)
        else:
            df[tox] = 0

    df = df.sort_values(['mrn', 'window_start']).reset_index(drop=True)
    print(f'  Loaded {len(df):,} window rows across {df["mrn"].nunique():,} patients')
    return df

grade_df = load_grade_data(GRADE_PATH)


In [ ]:
# ---- Cohort restriction: patients with LLM predictions (matches every other S2 panel) ----
llm_patients = pd.read_csv(LLM_PATIENT_PATH, encoding='latin-1', low_memory=False)
llm_patients['mrn'] = llm_patients['mrn'].apply(standardize_mrn)
llm_patients = llm_patients[llm_patients['mrn'].notna()].copy()
cohort_mrns = set(llm_patients['mrn'].drop_duplicates())
print(f'{len(cohort_mrns):,} unique patients with LLM predictions')

# ---- line1: first LOT per patient, same construction as S2B / S2C / S2D ----
covars = pd.read_csv(COVARS_PATH, low_memory=False)
covars['mrn'] = covars['mrn'].apply(standardize_mrn)
covars = covars[covars['mrn'].notna()].copy()
covars['lot'] = pd.to_numeric(covars['lot'], errors='coerce')
covars['lot_start'] = pd.to_datetime(covars['lot_start'], errors='coerce')
covars['censor_days'] = pd.to_numeric(covars['t_cutoff_lot'], errors='coerce')
covars = covars[covars['lot'].notna()].copy()

covars_valid = covars[covars['lot_start'].notna()].copy()
idx = covars_valid.sort_values(['mrn', 'lot']).groupby('mrn')['lot'].idxmin()
line1 = (covars_valid.loc[idx, ['mrn', 'lot', 'lot_start', 'censor_days']]
         .rename(columns={'lot': 'line1_lot', 'lot_start': 'line1_start'}))
line1 = line1[np.isfinite(line1['censor_days']) & (line1['censor_days'] > 0)].copy()
line1 = line1[line1['mrn'].isin(cohort_mrns)].copy()
line1['line1_end'] = line1['line1_start'] + pd.to_timedelta(line1['censor_days'], unit='D')

assert line1['mrn'].is_unique, 'more than one line-1 row per patient'
print(f'{len(line1):,} patients with line 1 dates and valid censoring (cohort-restricted)')
print(f"  line-1 LOT value distribution: {line1['line1_lot'].value_counts().sort_index().to_dict()}")

# ---- Follow-up scope: cohort restriction always; LOT-1 window truncation only if enabled ----
n_before_pts = grade_df['mrn'].nunique()
n_before_rows = len(grade_df)

grade_df = grade_df[grade_df['mrn'].isin(cohort_mrns)].copy()

if RESTRICT_TO_LOT1:
    grade_df = grade_df.merge(line1[['mrn', 'line1_start', 'line1_end']], on='mrn', how='inner')
    grade_df = grade_df[(grade_df['window_start'] >= grade_df['line1_start']) &
                        (grade_df['window_start'] <= grade_df['line1_end'])].copy()
    grade_df = grade_df.drop(columns=['line1_start', 'line1_end'])
    print('Follow-up: LOT-1 only')
else:
    print('Follow-up: all lines of therapy')

# Re-sort explicitly: extract_grade_flows reads each patient's grade sequence positionally,
# so chronological order within mrn is a correctness requirement, not a convenience.
grade_df = grade_df.sort_values(['mrn', 'window_start']).reset_index(drop=True)
assert grade_df.groupby('mrn')['window_start'].is_monotonic_increasing.all(), \
    'window_start not monotonic within patient'

print(f'  {n_before_pts:,} -> {grade_df["mrn"].nunique():,} patients '
      f'({n_before_rows:,} -> {len(grade_df):,} window rows)')


## Build the complete-follow-up G3+ cohort per toxicity

For each toxicity: find every patient who ever reached grade ≥3, then keep only those with
observed window data at *every* offset in the display horizon (their first G3+ window minus
`K_PRE` windows, through their last G3+ window plus `K_POST` windows). Each row records the grade
observed at each of those offsets.


In [ ]:
def extract_grade_flows(df, tox):
    if tox not in df.columns:
        return pd.DataFrame()
    sub = df[['mrn', tox]].rename(columns={tox: 'grade'})
    g3_mrns = sub.loc[sub['grade'] >= 3, 'mrn'].unique()
    if len(g3_mrns) == 0:
        return pd.DataFrame()
    sub = sub[sub['mrn'].isin(g3_mrns)]

    rows = []
    for mrn, grp in sub.groupby('mrn', sort=False):
        grades = grp['grade'].to_numpy()
        idxs = np.where(grades >= 3)[0]
        first_idx, last_idx = int(idxs[0]), int(idxs[-1])
        # Require full observed follow-up on both sides; else exclude the patient.
        if first_idx - K_PRE < 0 or last_idx + K_POST >= len(grades):
            continue
        rec = {'mrn': mrn, 'peak': int(grades.max())}
        for k in range(1, K_PRE + 1):
            rec[f'pre_{k}'] = int(grades[first_idx - k])
        for k in range(1, K_POST + 1):
            rec[f'post_{k}'] = int(grades[last_idx + k])
        rows.append(rec)
    return pd.DataFrame(rows)


all_flows = {}
for tox in TOXICITIES:
    flows = extract_grade_flows(grade_df, tox)
    all_flows[tox] = flows
    n = 0 if flows.empty else len(flows)
    print(f'  {TOX_DISPLAY[tox]}: complete-follow-up G3+ cohort n = {n:,}')


## Plot — 2×3 grid, one stacked-area panel per toxicity

Nature text tiers: 7pt panel titles/axis labels, 6pt tick labels, 5pt legend/percent annotations.
No `bbox_inches='tight'` on save.


In [ ]:
timepoints = [f'pre_{k}' for k in range(K_PRE, 0, -1)] + ['peak'] + [f'post_{k}' for k in range(1, K_POST + 1)]
x_days = np.arange(-K_PRE, K_POST + 1) * WINDOW_DAYS
x_tick_labels = [f'{int(d):+d}' if d != 0 else '0' for d in x_days]

fig = plt.figure(figsize=(7.5, 3.2))
rows, cols = 2, 3
items = list(all_flows.items())

for idx, (tox, flows_df) in enumerate(items):
    ax = fig.add_subplot(rows, cols, idx + 1)
    if flows_df.empty:
        ax.text(0.5, 0.5, f'{TOX_DISPLAY.get(tox, tox)}:\nno patients with complete\nfollow-up at \u00b1{K_PRE} windows',
                ha='center', va='center', transform=ax.transAxes, fontsize=6)
        ax.axis('off')
        continue

    total = len(flows_df)
    stack = np.zeros((len(GRADES_STACKED), len(timepoints)))
    for j, tp in enumerate(timepoints):
        obs = flows_df[tp].dropna().astype(int).to_numpy()
        for gi, g in enumerate(GRADES_STACKED):
            stack[gi, j] = 100.0 * (obs == g).sum() / total

    ax.stackplot(x_days, stack, colors=[GRADE_COLORS[g] for g in GRADES_STACKED],
                 alpha=0.90, edgecolor='white', linewidth=0.4)

    total_pct = stack.sum(axis=0)
    for j, val in enumerate(total_pct):
        ax.text(x_days[j], val + 2.5, f'{val:.0f}%', ha='center', va='bottom',
                 fontsize=5, color='#222', zorder=11)

    ax.axvline(0, color='#666', linestyle='--', linewidth=0.7, alpha=0.55, zorder=1)
    ax.set_xticks(x_days)
    ax.set_xticklabels(x_tick_labels, fontsize=6)
    ax.set_xlim(x_days[0] - 20, x_days[-1] + 20)
    ax.set_ylim(0, 110)
    ax.set_yticks([0, 25, 50, 75, 100])
    ax.tick_params(axis='y', labelsize=6)
    ax.set_title(f'{TOX_DISPLAY.get(tox, tox)} (n={total:,})', fontsize=7)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(True, axis='y', alpha=0.20, linewidth=0.4)

    if idx % cols == 0:
        ax.set_ylabel('% of G3+ cohort', fontsize=7)
    if idx >= (rows - 1) * cols or idx == len(items) - 1:
        ax.set_xlabel('Days from G3+ anchor (0 = peak)', fontsize=7)

legend_handles = [Rectangle((0, 0), 1, 1, facecolor=GRADE_COLORS[g], edgecolor='white') for g in GRADES_STACKED]
legend_labels = [GRADE_LABEL[g] for g in GRADES_STACKED]
fig.legend(legend_handles, legend_labels, loc='lower center', ncol=len(legend_labels),
           frameon=False, fontsize=5, bbox_to_anchor=(0.5, -0.01))

fig.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()

## Save PDF (no `bbox_inches='tight'`)


In [ ]:
fig.savefig(PDF_OUT, dpi=450)
print(f'Saved: {os.path.basename(PDF_OUT)}')
print()
print('Interpretation: at each 90-day offset the colored stack is the percent of the')
print('complete-follow-up G3+ cohort at grades G1-G5; the gap from the stack top to 100')
print('percent is the percent at grade 0 (no AE detected) at that offset; the peak column')
print('is 100 percent by construction since every included patient is G3+.')


## Export underlying values

Long format: one row per (toxicity, offset in days, grade), with the percent of that toxicity's
complete-follow-up G3+ cohort at that grade at that offset -- the exact values drawn in the
stacked-area plot above. Grade-0 percent (the gap to 100%) is included explicitly as `grade=0`
rather than left implicit, so the CSV is self-contained without needing to re-derive it from the
plot. `n_grade` is the count at that grade; `pct_ci_lower` / `pct_ci_upper` are the Wilson 95% CI
on `pct_of_cohort`. The figure is unchanged.


In [ ]:
from scipy.stats import binomtest

records = []
for tox, flows_df in all_flows.items():
    if flows_df.empty:
        continue
    total = len(flows_df)
    for j, tp in enumerate(timepoints):
        obs = flows_df[tp].dropna().astype(int).to_numpy()
        offset_days = int(x_days[j])
        for g in range(0, 6):  # include grade 0 explicitly
            k = int((obs == g).sum())
            pct = 100.0 * k / total
            ci = binomtest(k, total).proportion_ci(confidence_level=0.95, method='wilson')
            records.append({
                'toxicity': tox, 'toxicity_display': TOX_DISPLAY[tox],
                'offset_days': offset_days, 'timepoint_label': tp,
                'grade': g,
                'n_grade': k,
                'pct_of_cohort': round(pct, 3),
                'pct_ci_lower': round(100.0 * ci.low, 3),
                'pct_ci_upper': round(100.0 * ci.high, 3),
                'n_cohort': total,
            })

grade_dist_export = pd.DataFrame(records)
grade_dist_export.to_csv(CSV_OUT, index=False)
print(f'Saved: {os.path.basename(CSV_OUT)}')
grade_dist_export.head(12)
